# LangGraph Supervisor ? Agent ??

??????? Multi-Agent ????????? LangGraph supervisor demo???????`supervisor` ?????`researcher`?`writer`?`critic` ?????????????

## 1. ????

?? Notebook ???????? API Key?LangGraph ???????????? worker ????????????????

In [1]:
from __future__ import annotations

import argparse
import sys
from dataclasses import dataclass
from typing import Literal

from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict


if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

C:\Users\tallm\Documents\Codes\agent-building\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 2. ???????

`TASK` ???? Agent ????????`Role` ? `Route` ??????????????????? supervisor ???????

In [2]:
TASK = "??? 180 ?????????? AI Agent ???????"
Role = Literal["supervisor", "researcher", "writer", "critic"]
Route = Literal["researcher", "writer", "critic", "finish"]

## 3. AgentReport

?? worker ????? `AgentReport`?`passed` ?? Critic ??????????????

In [3]:
@dataclass(frozen=True)
class AgentReport:
    role: Role
    content: str
    passed: bool | None = None

## 4. SupervisorState

LangGraph ? State ????????????????????????????????????????????

In [4]:
class SupervisorState(TypedDict, total=False):
    task: str
    reports: list[AgentReport]
    evidence: list[str]
    draft: str
    critique: str
    approved: bool
    revision_count: int
    next_role: Route
    final_answer: str
    trace: list[str]

## 5. initial_state

??????????????????????????????????

In [5]:
def initial_state(task: str = TASK) -> SupervisorState:
    """???????????????????????"""
    return {
        "task": task,
        "reports": [],
        "evidence": [],
        "draft": "",
        "critique": "",
        "approved": False,
        "revision_count": 0,
        "next_role": "researcher",
        "final_answer": "",
        "trace": [],
    }

## 6. latest_report

???????????????????Supervisor ???? Writer ??????? Critic?

In [6]:
def latest_report(state: SupervisorState, role: Role) -> AgentReport | None:
    """?????????????????"""
    for report in reversed(state.get("reports", [])):
        if report.role == role:
            return report
    return None

## 7. choose_next_role

?? supervisor ????????????????????? Critic ??????? Writer ????????????????

In [7]:
def choose_next_role(state: SupervisorState) -> Route:
    """Supervisor ????????????????? worker?"""
    if not state.get("evidence"):
        return "researcher"
    if not state.get("draft"):
        return "writer"
    if state.get("approved"):
        return "finish"

    last = state.get("reports", [])[-1]
    if last.role == "writer":
        return "critic"
    if last.role == "critic" and state.get("revision_count", 0) >= 2:
        return "finish"
    if last.role == "critic":
        return "writer"
    if state.get("revision_count", 0) >= 2:
        return "finish"
    return "writer"

## 8. append_trace

????????????????????????????????????

In [8]:
def append_trace(state: SupervisorState, event: str) -> list[str]:
    """???? trace ??????????????"""
    return [*state.get("trace", []), event]

## 9. append_report

? trace ???reports ???????????????????????????????????

In [9]:
def append_report(state: SupervisorState, report: AgentReport) -> list[AgentReport]:
    """???? reports ???????????????"""
    return [*state.get("reports", []), report]

## 10. supervisor_node

Supervisor ????????????????????????????? supervisor ??????

In [10]:
def supervisor_node(state: SupervisorState) -> SupervisorState:
    """???????????????????"""
    route = choose_next_role(state)
    update: SupervisorState = {
        "next_role": route,
        "trace": append_trace(state, f"supervisor -> {route}"),
    }
    if route == "finish":
        update["final_answer"] = state.get("draft", "")
    return update

## 11. researcher_node

Researcher ?????????????????????

In [11]:
def researcher_node(state: SupervisorState) -> SupervisorState:
    """Researcher ??????????????"""
    evidence = [
        "E1: LLM ?????????????????????????????",
        "E2: ???????????????????????? Agent ???",
        "E3: ??????????????????????????????",
    ]
    content = "?????\n" + "\n".join(evidence)
    report = AgentReport(role="researcher", content=content)
    return {
        "evidence": evidence,
        "reports": append_report(state, report),
        "trace": append_trace(state, "researcher: collected 3 evidence items"),
    }

## 12. writer_node

Writer ???????????? ID ????? Critic ????????????? `[E1]`?`[E2]`?`[E3]`?

In [12]:
def writer_node(state: SupervisorState) -> SupervisorState:
    """Writer ????????????????????? critic ?????"""
    revision_count = state.get("revision_count", 0) + 1
    if revision_count == 1:
        draft = (
            "AI Agent ??????????????????????????"
            "?????????????????????????????????????"
            "Agent ????????????????????????????????"
        )
    else:
        draft = (
            "AI Agent ??????????????????????????????????[E1]?"
            "???????????????????????[E2]?"
            "??????????????? Agent ??????????[E3]?"
        )

    report = AgentReport(role="writer", content=draft)
    return {
        "draft": draft,
        "revision_count": revision_count,
        "reports": append_report(state, report),
        "trace": append_trace(state, f"writer: produced draft v{revision_count}"),
    }

## 13. critic_node

Critic ?????????????????????????? 180 ??????????????

In [13]:
def critic_node(state: SupervisorState) -> SupervisorState:
    """Critic ?????????????????"""
    draft = state.get("draft", "")
    missing = [tag for tag in ("[E1]", "[E2]", "[E3]") if tag not in draft]
    too_long = len(draft) > 180
    approved = not missing and not too_long

    if approved:
        critique = "??????? E1/E2/E3 ????????? 180 ????"
    else:
        problems = []
        if missing:
            problems.append(f"??????: {', '.join(missing)}")
        if too_long:
            problems.append("?? 180 ???")
        critique = "????" + "?".join(problems) + "?? Writer ???"

    report = AgentReport(role="critic", content=critique, passed=approved)
    return {
        "critique": critique,
        "approved": approved,
        "reports": append_report(state, report),
        "trace": append_trace(state, f"critic: {'approved' if approved else 'requested revision'}"),
    }

## 14. route_after_supervisor

LangGraph ????????? `next_role`????????? `supervisor_node` ????

In [14]:
def route_after_supervisor(state: SupervisorState) -> Route:
    """LangGraph ????? supervisor ??? next_role?"""
    return state["next_role"]

## 15. build_graph

??? supervisor ??? worker ?????? worker ?????? supervisor?? supervisor ???????????

In [15]:
def build_graph():
    """?? LangGraph supervisor ???"""
    graph = StateGraph(SupervisorState)
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("researcher", researcher_node)
    graph.add_node("writer", writer_node)
    graph.add_node("critic", critic_node)

    graph.add_edge(START, "supervisor")
    graph.add_conditional_edges(
        "supervisor",
        route_after_supervisor,
        {
            "researcher": "researcher",
            "writer": "writer",
            "critic": "critic",
            "finish": END,
        },
    )
    graph.add_edge("researcher", "supervisor")
    graph.add_edge("writer", "supervisor")
    graph.add_edge("critic", "supervisor")
    return graph.compile()

## 16. run_supervisor

??????????????Notebook ??????

In [16]:
def run_supervisor(task: str = TASK) -> SupervisorState:
    """??????? supervisor ? Agent ???"""
    app = build_graph()
    return app.invoke(initial_state(task))

## 17. print_trace

???????????????????????????

In [17]:
def print_trace(state: SupervisorState) -> None:
    """?? Agent ???????????????"""
    print(f"???{state['task']}\n")
    print("=== ???? ===")
    for event in state["trace"]:
        print(f"- {event}")

    print("\n=== ???? ===")
    for report in state["reports"]:
        suffix = "" if report.passed is None else f" | passed={report.passed}"
        print(f"\n[{report.role}{suffix}]\n{report.content}")

    print("\n=== ???? ===")
    print(state["final_answer"])

## 18. graph_to_mermaid

?? Mermaid ??????????????? supervisor ????

In [18]:
def graph_to_mermaid() -> str:
    """?? LangGraph ???????? Mermaid ???"""
    return build_graph().get_graph(xray=True).draw_mermaid()

## 19. explain

?? supervisor ????????

In [19]:
def explain() -> None:
    print(
        "\n".join([
            "Supervisor ???? Agent ???????????? worker?",
            "Supervisor ???????????????????????",
            "Worker ????????Researcher ????Writer ???Critic ???",
            "?????????????????????????????????",
            "????????????????? worker ??????????????",
        ])
    )

## 20. self_test

????????????????????????????????????

In [20]:
def self_test() -> None:
    state = run_supervisor()
    assert state["approved"] is True
    assert state["revision_count"] == 2
    assert "[E1]" in state["final_answer"]
    assert "[E2]" in state["final_answer"]
    assert "[E3]" in state["final_answer"]
    assert len(state["final_answer"]) <= 180
    assert "supervisor -> researcher" in state["trace"]
    assert "supervisor -> writer" in state["trace"]
    assert "supervisor -> critic" in state["trace"]
    graph = graph_to_mermaid()
    assert "supervisor" in graph
    assert "researcher" in graph
    assert "writer" in graph
    assert "critic" in graph
    print("? self-test passed: supervisor routed workers, revision loop converged, graph is valid.")

## 21. main

???????? Notebook ??????? `.py` ???? CLI ???Notebook ??????????? cell?

In [21]:
def main() -> None:
    parser = argparse.ArgumentParser(description="LangGraph Supervisor ??? Agent ??")
    parser.add_argument("--graph", action="store_true", help="?? Mermaid ???")
    parser.add_argument("--explain", action="store_true", help="????????")
    parser.add_argument("--self-test", action="store_true", help="????????")
    args = parser.parse_args()

    if args.graph:
        print(graph_to_mermaid())
    elif args.explain:
        explain()
    elif args.self_test:
        self_test()
    else:
        print_trace(run_supervisor())

## 22. ??????

????? LangGraph ? supervisor ????????

In [22]:
print(graph_to_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	writer(writer)
	critic(critic)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	critic --> supervisor;
	researcher --> supervisor;
	supervisor -. &nbsp;finish&nbsp; .-> __end__;
	supervisor -.-> critic;
	supervisor -.-> researcher;
	supervisor -.-> writer;
	writer --> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 23. ?? demo

?? supervisor ???? researcher?writer?critic??? Critic ???????? Writer ???

In [23]:
state = run_supervisor()
print_trace(state)

?????? 180 ?????????? AI Agent ???????

=== ???? ===
- supervisor -> researcher
- researcher: collected 3 evidence items
- supervisor -> writer
- writer: produced draft v1
- supervisor -> critic
- critic: requested revision
- supervisor -> writer
- writer: produced draft v2
- supervisor -> critic
- critic: approved
- supervisor -> finish

=== ???? ===

[researcher]
?????
E1: LLM ?????????????????????????????
E2: ???????????????????????? Agent ???
E3: ??????????????????????????????

[writer]
AI Agent ???????????????????????????????????????????????????????????????Agent ????????????????????????????????

[critic | passed=False]
??????????: [E1], [E2], [E3]?? Writer ???

[writer]
AI Agent ??????????????????????????????????[E1]????????????????????????[E2]???????????????? Agent ??????????[E3]?

[critic | passed=True]
??????? E1/E2/E3 ????????? 180 ????

=== ???? ===
AI Agent ??????????????????????????????????[E1]????????????????????????[E2]???????????????? Agent ??????????[E3]?


## 24. ????

???????????? Notebook ????????

In [24]:
self_test()

? self-test passed: supervisor routed workers, revision loop converged, graph is valid.
